# Full estimates, organized by question
Tight settings: strict contraction tolerance, many random starting points and many optimizer iterations. Same 16 models as `pset_quick_estimates_by_question.ipynb`; only the settings in section 0 differ.

Start the kernel with several threads (`-t auto`). The best of the starting points is kept, and starts that error are logged next to each estimate as `*_failed_starts.jdl2`. Results go to `estimates_full/`. The older `models_*` folders are not touched.

**How to read this notebook:** one section per question. Each model is a three-line cell: copy the base dataset, set the instruments `Z`, estimate. The `Z = ...` line is the only thing that changes between models.

Notation: `X` = constant, x1, x2. `rival X` = BLP instruments (sum of the rivals' x1, x2 in the same market). `Hausman` = average price of the same product in the other markets. `price` = the product's own price used as its own instrument. `w` = cost shifter. `GH` = Gandhi-Houde instruments.

### The 16 models
| # | Question | Model | Instruments Z | Dataset | Saved as |
|---|---|---|---|---|---|
| 1 | 8c | BLP instruments only | X, rival X | T=100, J=3 | `estimates_full/q8c_blp_only_T100_J3.jdl2` |
| 2 | 8c | BLP instruments + Hausman | X, Hausman, rival X | T=100, J=3 | `estimates_full/q8c_blp_hausman_T100_J3.jdl2` |
| 3 | 8e | BLP instruments only | X, rival X | T=10, J=3 | `estimates_full/q8e_blp_only_T10_J3.jdl2` |
| 4 | 8e | BLP instruments + Hausman | X, Hausman, rival X | T=10, J=3 | `estimates_full/q8e_blp_hausman_T10_J3.jdl2` |
| 5 | 8f | BLP instruments + price | X, price, rival X | T=100, J=3 | `estimates_full/q8f_blp_price_T100_J3.jdl2` |
| 6 | 8f | BLP instruments + price + Hausman | X, price, Hausman, rival X | T=100, J=3 | `estimates_full/q8f_blp_price_hausman_T100_J3.jdl2` |
| 7 | 9a | BLP instruments + w | X, rival X, w | T=100, J=3 | `estimates_full/q9a_blp_w_T100_J3.jdl2` |
| 8 | 9a | BLP instruments + Hausman + w | X, Hausman, rival X, w | T=100, J=3 | `estimates_full/q9a_blp_hausman_w_T100_J3.jdl2` |
| 9 | 9b | BLP instruments + w | X, rival X, w | T=10, J=3 | `estimates_full/q9b_blp_w_T10_J3.jdl2` |
| 10 | 9b | BLP instruments + Hausman + w | X, Hausman, rival X, w | T=10, J=3 | `estimates_full/q9b_blp_hausman_w_T10_J3.jdl2` |
| 11 | 9e | Demand + supply, Bertrand | demand: X, Hausman, rival X, w; supply: X, rival X, Z, w | T=100, J=3 | `estimates_full/q9e_system_bertrand_T100_J3.jdl2` |
| 12 | 9e | Demand + supply, Competition | same as Bertrand | T=100, J=3 | `estimates_full/q9e_system_competition_T100_J3.jdl2` |
| 13 | 9e | Demand + supply, Collusion | same as Bertrand | T=100, J=3 | `estimates_full/q9e_system_collusion_T100_J3.jdl2` |
| 14 | 11a | BLP instruments only | X, rival X | T=100, J=5 | `estimates_full/q11a_blp_only_T100_J5.jdl2` |
| 15 | 11a | GH instruments only | X, GH1, GH2 | T=100, J=5 | `estimates_full/q11a_gh_only_T100_J5.jdl2` |
| 16 | 11a | GH instruments + Hausman | X, Hausman, GH1, GH2 | T=100, J=5 | `estimates_full/q11a_gh_hausman_T100_J5.jdl2` |

## 0. Setup

In [ ]:
using MAT, DataFrames, Statistics, LinearAlgebra, JLD2
include("blp_optimize/blp_new.jl")
using .blp
Threads.nthreads()   # starting points run in parallel, one per thread

In [ ]:
# ---- Solver settings (FULL) ----
# The quick notebook uses: toler = 1e-4, max_iter = 1000, Ns = 1, iterations = 15
SOLVER = (
    toler      = 1e-10,   # tolerance of the inner contraction
    max_iter   = 10_000,  # max contraction iterations
    Ns         = 16,      # number of random starting points for sigma
    iterations = 100,     # optimizer iterations per starting point
    time_limit = 600.0,   # seconds per start; a start that hits it returns its best point so far
    sigma_lower = 0.0,    # lower bound of sigma (was 0.1 by default)
)

mkpath("estimates_full")

# ---- True parameters of the simulation ----
beta = [5, 1, 1]; alpha_bar = -1; gamma = [2, 1, 1]; sigma = 1.0

### Helper functions
- `process_data(d, errors)` builds a `BLPDemand` model from a simulated dataset.
- `set_Z!(m, Z)` sets the instruments and the weighting matrix `W = (Z'Z)^-1`.
- `estimate!(m, question, label, file)` estimates, saves to `file`, and records the result for the summary tables.

In [ ]:
function process_data(d, errors)
    J, T = size(d["P_opt"])
    p      = reshape(d["P_opt"], (T*J, 1))
    shares = reshape(d["shares"], (T*J, 1))
    data   = blp.BLPData(T, J, d["x1"], p, shares, errors)
    return blp.BLPDemand(BLPData = data)
end

set_Z!(m, Z) = (m.Z = Z; m.W = inv(Z' * Z); m)

results = []   # (question, label, model, seconds), filled by estimate!

function estimate!(model, question, label, file)
    println("[$question] $label ..."); flush(stdout)
    failures = []   # starts that errored; the search continues with the other starts
    t = @elapsed if model isa blp.BLPDemand
        blp.estimate_blp!(model; SOLVER..., failures = failures)
    else
        blp.estimate_blp_joint!(model; SOLVER..., failures = failures)
    end
    jldsave(file; model = model)
    jldsave(replace(file, ".jdl2" => "_failed_starts.jdl2"); failures = failures)
    println("    $(length(failures)) of $(SOLVER.Ns) starts failed, $(round(t, digits = 1))s")
    println("    params = ", round.(vec(model.params), digits = 4))
    push!(results, (question, label, model, t))
    return model
end

### Tables
`section_table(question)` shows the estimates of one question next to the true values (`intercept, x1, x2, price` are the linear demand parameters; `sigma` is the random-coefficient parameter).

In [ ]:
demand_of(m) = m isa blp.BLP ? m.BLPDemand : m

function results_table(rows)
    DataFrame(
        question = [r[1] for r in rows],
        model    = [r[2] for r in rows],
        intercept = [demand_of(r[3]).beta[1] for r in rows],
        x1       = [demand_of(r[3]).beta[2] for r in rows],
        x2       = [demand_of(r[3]).beta[3] for r in rows],
        price    = [demand_of(r[3]).beta[4] for r in rows],
        sigma    = [r[3].params[end] for r in rows],
        seconds  = [round(r[4], digits = 1) for r in rows],
    )
end

truth = DataFrame(question = "true", model = "true values", intercept = beta[1], x1 = beta[2], x2 = beta[3], price = alpha_bar, sigma = sigma, seconds = 0.0)

section_table(q) = vcat(truth, results_table(filter(r -> r[1] == q, results)))

## 1. Data and instruments
Three simulated datasets. For each one we build the base model and the instruments that are shared by several models. Every model below starts from a `deepcopy` of its base model, so models never share state.

In [ ]:
d_T10_J3  = matread("../build/input/public/simulation/10markets3products.mat")
d_T100_J3 = matread("../build/input/public/simulation/100markets3products.mat")
d_T100_J5 = matread("../build/input/public/simulation/100markets5products.mat")

**T = 100, J = 3** (also the source of the simulation draws used for J = 5).

In [ ]:
errors_T100 = log.(d_T100_J3["alphas"] .- 1)   # simulation draws of the random coefficient

base_T100_J3 = process_data(d_T100_J3, errors_T100)

price_bar_T100_J3  = blp.compute_hausman(base_T100_J3)   # Hausman: price of the same product in other markets
X_rivals_T100_J3   = blp.compute_blp(base_T100_J3)       # BLP: sum of rivals' x1, x2 in the same market
w_T100_J3          = d_T100_J3["w"]                      # cost shifter

**T = 10, J = 3.**

In [ ]:
base_T10_J3 = process_data(d_T10_J3, log.(d_T10_J3["alphas"] .- 1))

price_bar_T10_J3 = blp.compute_hausman(base_T10_J3)
X_rivals_T10_J3  = blp.compute_blp(base_T10_J3)
w_T10_J3         = d_T10_J3["w"]

**T = 100, J = 5.** As in `pset_io.ipynb`, the draws of the random coefficient are taken from the J = 3 dataset, so that the J = 3 and J = 5 results are comparable.

In [ ]:
base_T100_J5 = process_data(d_T100_J5, errors_T100)

price_bar_T100_J5 = blp.compute_hausman(base_T100_J5)
X_rivals_T100_J5  = blp.compute_blp(base_T100_J5)

# Question 8: BLP estimation with price as an endogenous variable

## 8c) T = 100, J = 3
Estimates with the BLP instruments only, and then adding the Hausman instrument.

**Model 1.** Z = [X, rival X]

In [ ]:
m8c_blp_only = deepcopy(base_T100_J3)
set_Z!(m8c_blp_only, hcat(m8c_blp_only.X, X_rivals_T100_J3))
estimate!(m8c_blp_only, "8c", "BLP instruments only, T=100 J=3", "estimates_full/q8c_blp_only_T100_J3.jdl2");

**Model 2.** Z = [X, Hausman, rival X]

In [ ]:
m8c_blp_hausman = deepcopy(base_T100_J3)
set_Z!(m8c_blp_hausman, hcat(m8c_blp_hausman.X, price_bar_T100_J3, X_rivals_T100_J3))
estimate!(m8c_blp_hausman, "8c", "BLP instruments + Hausman, T=100 J=3", "estimates_full/q8c_blp_hausman_T100_J3.jdl2");

In [ ]:
section_table("8c")

## 8e) T = 10, J = 3
Same two specifications on the small dataset.

**Model 3.** Z = [X, rival X]

In [ ]:
m8e_blp_only = deepcopy(base_T10_J3)
set_Z!(m8e_blp_only, hcat(m8e_blp_only.X, X_rivals_T10_J3))
estimate!(m8e_blp_only, "8e", "BLP instruments only, T=10 J=3", "estimates_full/q8e_blp_only_T10_J3.jdl2");

**Model 4.** Z = [X, Hausman, rival X]

In [ ]:
m8e_blp_hausman = deepcopy(base_T10_J3)
set_Z!(m8e_blp_hausman, hcat(m8e_blp_hausman.X, price_bar_T10_J3, X_rivals_T10_J3))
estimate!(m8e_blp_hausman, "8e", "BLP instruments + Hausman, T=10 J=3", "estimates_full/q8e_blp_hausman_T10_J3.jdl2");

In [ ]:
section_table("8e")

## 8f) T = 100, J = 3, price used as its own instrument
`X_all` is X plus the product's own price, so price is treated as exogenous. This is the naive benchmark.

**Model 5.** Z = [X, price, rival X]

In [ ]:
m8f_blp_price = deepcopy(base_T100_J3)
set_Z!(m8f_blp_price, hcat(m8f_blp_price.X_all, X_rivals_T100_J3))
estimate!(m8f_blp_price, "8f", "BLP instruments + price, T=100 J=3", "estimates_full/q8f_blp_price_T100_J3.jdl2");

**Model 6.** Z = [X, price, Hausman, rival X]

In [ ]:
m8f_blp_price_hausman = deepcopy(base_T100_J3)
set_Z!(m8f_blp_price_hausman, hcat(m8f_blp_price_hausman.X_all, price_bar_T100_J3, X_rivals_T100_J3))
estimate!(m8f_blp_price_hausman, "8f", "BLP instruments + price + Hausman, T=100 J=3", "estimates_full/q8f_blp_price_hausman_T100_J3.jdl2");

In [ ]:
section_table("8f")

# Question 9: adding the cost shifter w as an instrument, and the supply side

## 9a) T = 100, J = 3
The cost shifter `w` is added to the instruments.

**Model 7.** Z = [X, rival X, w]

In [ ]:
m9a_blp_w = deepcopy(base_T100_J3)
set_Z!(m9a_blp_w, hcat(m9a_blp_w.X, X_rivals_T100_J3, w_T100_J3))
estimate!(m9a_blp_w, "9a", "BLP instruments + w, T=100 J=3", "estimates_full/q9a_blp_w_T100_J3.jdl2");

**Model 8.** Z = [X, Hausman, rival X, w]

In [ ]:
m9a_blp_hausman_w = deepcopy(base_T100_J3)
set_Z!(m9a_blp_hausman_w, hcat(m9a_blp_hausman_w.X, price_bar_T100_J3, X_rivals_T100_J3, w_T100_J3))
estimate!(m9a_blp_hausman_w, "9a", "BLP instruments + Hausman + w, T=100 J=3", "estimates_full/q9a_blp_hausman_w_T100_J3.jdl2");

In [ ]:
section_table("9a")

## 9b) T = 10, J = 3
Same as 9a on the small dataset.

**Model 9.** Z = [X, rival X, w]

In [ ]:
m9b_blp_w = deepcopy(base_T10_J3)
set_Z!(m9b_blp_w, hcat(m9b_blp_w.X, X_rivals_T10_J3, w_T10_J3))
estimate!(m9b_blp_w, "9b", "BLP instruments + w, T=10 J=3", "estimates_full/q9b_blp_w_T10_J3.jdl2");

**Model 10.** Z = [X, Hausman, rival X, w]

In [ ]:
m9b_blp_hausman_w = deepcopy(base_T10_J3)
set_Z!(m9b_blp_hausman_w, hcat(m9b_blp_hausman_w.X, price_bar_T10_J3, X_rivals_T10_J3, w_T10_J3))
estimate!(m9b_blp_hausman_w, "9b", "BLP instruments + Hausman + w, T=10 J=3", "estimates_full/q9b_blp_hausman_w_T10_J3.jdl2");

In [ ]:
section_table("9b")

## 9e) T = 100, J = 3: demand and supply estimated jointly
The ownership matrix `Omega` encodes conduct:
- **Bertrand:** identity (each product maximizes its own profit).
- **Competition:** zeros (price equals marginal cost).
- **Collusion:** ones (all products maximize joint profit).

Demand instruments are the ones of Model 8: Z = [X, Hausman, rival X, w].
Supply side: cost = const + Z + w, with instruments [X, rival X, Z, w], where `Z` is the cost shifter from the dataset.

In [ ]:
# Demand side, shared by the three conduct assumptions
demand_T100_J3 = deepcopy(base_T100_J3)
set_Z!(demand_T100_J3, hcat(demand_T100_J3.X, price_bar_T100_J3, X_rivals_T100_J3, w_T100_J3))
demand_T100_J3.M = blp.gmm_M(demand_T100_J3.X_all, demand_T100_J3.Z, demand_T100_J3.W)   # residual maker, needed by the joint estimator

# Supply side, shared by the three conduct assumptions
X_supply = hcat(ones(base_T100_J3.T * base_T100_J3.J, 1), d_T100_J3["Z"], w_T100_J3)                              # const, cost shifter Z, w
Z_supply = hcat(base_T100_J3.X, X_rivals_T100_J3, X_supply[:, 2:end])                      # X, rival X, Z, w
supply_T100_J3 = blp.BLPSupply(X = X_supply, Z = Z_supply, W = inv(Z_supply' * Z_supply))

**Model 11.** Bertrand: `Omega = I`

In [ ]:
demand_bertrand = deepcopy(demand_T100_J3)
demand_bertrand.Omega = Matrix{Float64}(I, 3, 3)
m9e_bertrand = blp.BLP(BLPDemand = demand_bertrand, BLPSupply = deepcopy(supply_T100_J3))
estimate!(m9e_bertrand, "9e", "Demand + supply, Bertrand, T=100 J=3", "estimates_full/q9e_system_bertrand_T100_J3.jdl2");

**Model 12.** Competition: `Omega = 0`

In [ ]:
demand_competition = deepcopy(demand_T100_J3)
demand_competition.Omega = zeros(3, 3)
m9e_competition = blp.BLP(BLPDemand = demand_competition, BLPSupply = deepcopy(supply_T100_J3))
estimate!(m9e_competition, "9e", "Demand + supply, Competition, T=100 J=3", "estimates_full/q9e_system_competition_T100_J3.jdl2");

**Model 13.** Collusion: `Omega = 1`

In [ ]:
demand_collusion = deepcopy(demand_T100_J3)
demand_collusion.Omega = ones(3, 3)
m9e_collusion = blp.BLP(BLPDemand = demand_collusion, BLPSupply = deepcopy(supply_T100_J3))
estimate!(m9e_collusion, "9e", "Demand + supply, Collusion, T=100 J=3", "estimates_full/q9e_system_collusion_T100_J3.jdl2");

Demand parameters of the three systems, and the supply-side parameters (true cost parameters: const = 2, Z = 1, w = 1).

In [ ]:
section_table("9e")

In [ ]:
DataFrame(
    model = [r[2] for r in results if r[1] == "9e"],
    cost_intercept = [r[3].params[5] for r in results if r[1] == "9e"],
    cost_Z     = [r[3].params[6] for r in results if r[1] == "9e"],
    cost_w     = [r[3].params[7] for r in results if r[1] == "9e"],
)

# Question 11: more products (T = 100, J = 5) and Gandhi-Houde instruments

## 11a) Gandhi-Houde (GH) instruments
For each market and each characteristic x1, x2:
- **GH1** = number of rival products whose characteristic is within one standard deviation (of that characteristic in the whole sample) of the product's own value.
- **GH2** = sum over rivals of the squared difference between the characteristics.

`gh1` and `gh2` both have one column per characteristic (x1, x2).

In [ ]:
T = 100; J = 5
X_J5 = base_T100_J5.X

gh1 = zeros(T*J, 2)   # columns: x1, x2
gh2 = zeros(T*J, 2)

for col in 2:3   # skip the constant (column 1 of X)
    cutoff = std(X_J5[:, col])
    for t in 1:T
        rows = (t - 1) * J + 1 : t * J
        feat = X_J5[rows, col]
        gh1[rows, col - 1] = sum(abs.(feat .- feat') .< cutoff, dims = 2) .- 1   # -1 removes the product itself
        gh2[rows, col - 1] = sum(abs.(feat .- feat') .^ 2, dims = 2)
    end
end

**Model 14.** Z = [X, rival X]

In [ ]:
m11a_blp_only = deepcopy(base_T100_J5)
set_Z!(m11a_blp_only, hcat(m11a_blp_only.X, X_rivals_T100_J5))
estimate!(m11a_blp_only, "11a", "BLP instruments only, T=100 J=5", "estimates_full/q11a_blp_only_T100_J5.jdl2");

**Model 15.** Z = [X, GH1, GH2]

In [ ]:
m11a_gh_only = deepcopy(base_T100_J5)
set_Z!(m11a_gh_only, hcat(m11a_gh_only.X, gh1, gh2))
estimate!(m11a_gh_only, "11a", "GH instruments only, T=100 J=5", "estimates_full/q11a_gh_only_T100_J5.jdl2");

**Model 16.** Z = [X, Hausman, GH1, GH2]

In [ ]:
m11a_gh_hausman = deepcopy(base_T100_J5)
set_Z!(m11a_gh_hausman, hcat(m11a_gh_hausman.X, price_bar_T100_J5, gh1, gh2))
estimate!(m11a_gh_hausman, "11a", "GH instruments + Hausman, T=100 J=5", "estimates_full/q11a_gh_hausman_T100_J5.jdl2");

In [ ]:
section_table("11a")

# Summary of all 16 models

In [ ]:
vcat(truth, results_table(results))

Standard errors (demand parameters and sigma). Models 11-13 are the joint demand + supply systems.

In [ ]:
function se_table(rows)
    # Var is ordered as [beta (const, x1, x2, price); ...; sigma], so the first 4 entries are the demand betas
    # and the last one is sigma. Systems carry the supply parameters in between.
    se(m) = [v >= 0 ? sqrt(v) : NaN for v in diag(m.Var)]   # NaN = negative variance, i.e. a singular (unreliable) Var
    DataFrame(
        question = [r[1] for r in rows],
        model    = [r[2] for r in rows],
        se_intercept = [se(r[3])[1] for r in rows],
        se_x1    = [se(r[3])[2] for r in rows],
        se_x2    = [se(r[3])[3] for r in rows],
        se_price = [se(r[3])[4] for r in rows],
        se_sigma = [se(r[3])[end] for r in rows],
    )
end
se_table(results)